# 📊 Sales Analysis - AAL (Australia Apparel Ltd)

Fourth Quarter Sales Analysis (2020)

---

## 📝Project Statement
🔍 AAL is analyzing its Q4 sales data across Australian states to:
1. Identify high revenue states
2. Improve low-performing regions
3. Enable data-driven expansion decisions


In [19]:
# Import libraries
import pandas as pd
import plotly.express as px

In [20]:
# Load dataset into a pandas dataframe
df = pd.read_csv('./Data/AusApparalSales4thQrt2020.csv')

In [21]:
## Analyzing the data 
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7560 entries, 0 to 7559
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Date    7560 non-null   object
 1   Time    7560 non-null   object
 2   State   7560 non-null   object
 3   Group   7560 non-null   object
 4   Unit    7560 non-null   int64 
 5   Sales   7560 non-null   int64 
dtypes: int64(2), object(4)
memory usage: 354.5+ KB


## 1. Data Wrangling

#### 💩 Following Data issue has been identified
- 📅 `Date` column is not in date type it is object type
- 📝`Time`, `State` and `Group` column has extra space in it.

##### Step 1:🫧 Cleaning the "Time", "State", and "Group" columns which have extra spaces. Remove the extra spaces in the columns

In [22]:
df['Time'] = df['Time'].str.strip()
df['State'] = df['State'].str.strip()
df['Group'] = df['Group'].str.strip()

##### Step 2: 📅 Update the Date column data type

In [23]:
df['Date'] = pd.to_datetime(df['Date'])

In [24]:
# Check missing values
df.isna().sum()

Date     0
Time     0
State    0
Group    0
Unit     0
Sales    0
dtype: int64

### 🔭 OBSERVATION: There is no missing value so nothing to fill in of drop the records


## 📉 2. Data Analysis

In [25]:
# Descriptive statistics
df[['Sales','Unit']].describe()

,Sales,Unit
count,7560.000000,7560.000000
mean,45013.558201,18.005423
std,32253.506944,12.901403
min,5000.000000,2.000000
25%,20000.000000,8.000000
50%,35000.000000,14.000000
75%,65000.000000,26.000000
max,162500.000000,65.000000


#### 2.a: Perform descriptive statistical analysis on the data in the Sales and Unit columns. 
 - Utilize techniques such as `mean`, `median`, `mode`, and `standard deviation` for this analysis. 

In [26]:
# Mean, Median, Mode, Std
stats_sales = {
    'Mean': df['Sales'].mean(),
    'Median': df['Sales'].median(),
    'Mode': df['Sales'].mode()[0],
    'Std': df['Sales'].std()
}
stats_unit = {
    'Mean': df['Unit'].mean(),
    'Median': df['Unit'].median(),
    'Mode': df['Unit'].mode()[0],
    'Std': df['Unit'].std()
}
print("💵 Sales Statistics:")
print(f"  Mean: { round(stats_sales['Mean'], 2)}")
print(f"  Median: { round(stats_sales['Median'], 2)}")
print(f"  Mode: {stats_sales['Mode']}")
print(f"  Std: {round(stats_sales['Std'], 2)}")
print("")
print("📦 Unit Statistics:")
print(f"  Mean: { round(stats_unit['Mean'], 2)}")
print(f"  Median: {round(stats_unit['Median'], 2)}")
print(f"  Mode: {stats_unit['Mode']}")
print(f"  Std: {round(stats_unit['Std'], 2)}")

💵 Sales Statistics:
  Mean: 45013.56
  Median: 35000.0
  Mode: 22500
  Std: 32253.51

📦 Unit Statistics:
  Mean: 18.01
  Median: 14.0
  Mode: 9
  Std: 12.9


#### 2.b : Identify the group with the highest sales and the group with the lowest sales based on the data provided. 
#### 2.c : Identify the group with the highest and lowest sales based on the data provided.

In [27]:
# Highest and lowest sales states
state_group = df.groupby('State')['Sales'].sum()
highest_state = state_group.idxmax()
lowest_state = state_group.idxmin()

print(f"📊 Highest Sales State: {highest_state} with ${state_group[highest_state]:,.2f}")
print(f"📉 Lowest Sales State: {lowest_state} with ${state_group[lowest_state]:,.2f}")

📊 Highest Sales State: VIC with $105,565,000.00
📉 Lowest Sales State: WA with $22,152,500.00


#### 2.d. Generate weekly, monthly, and quarterly reports to document and present the results of the analysis conducted.

In [28]:
# Setting Date as index for time series analysis
df.set_index('Date', inplace=True)

In [29]:
# Populate the weekly sales matrix

weekly_df = df.resample('W')['Sales'].sum().apply(lambda x: f"${x:,.2f}")
weekly_df.name = 'Weekly Sales'
weekly_df= weekly_df.to_frame(name='Weekly Sales($)')

print(f"📊 Weekly Sales Data:")
print(weekly_df.head())

# Calculate monthly sales
monthly_df = df.resample('ME')['Sales'].sum().apply(lambda x: f"${x:,.2f}")
monthly_df.name = 'Monthly Sales'
monthly_df = monthly_df.to_frame(name='Monthly Sales($)')

print(f"\n📊 Monthly Sales Data:")
print(monthly_df.head())


# Quarterly sales
quarterly_df = df.resample('QE')['Sales'].sum().apply(lambda x: f"${x:,.2f}")
quarterly_df.name = 'Quarterly Sales'
quarterly_df = quarterly_df.to_frame(name='Quarterly Sales($)')

print(f"\n📊 Quarterly Sales Data:")
print(quarterly_df.head()) 



📊 Weekly Sales Data:
           Weekly Sales($)
Date                      
2020-10-04  $15,045,000.00
2020-10-11  $27,002,500.00
2020-10-18  $26,640,000.00
2020-10-25  $26,815,000.00
2020-11-01  $21,807,500.00

📊 Monthly Sales Data:
           Monthly Sales($)
Date                       
2020-10-31  $114,290,000.00
2020-11-30   $90,682,500.00
2020-12-31  $135,330,000.00

📊 Quarterly Sales Data:
           Quarterly Sales($)
Date                         
2020-12-31    $340,302,500.00


## 3. Data Visualization

### #️⃣ Write all the methods required for builing the dashboard

#### 📊 State-wise sales analysis 

In [30]:
def generate_sales_donut_matrix(sales_df, group_by_column):
    """
    Generates a sales matrix for each state and visualizes it as a donut chart.
    """
    state_group_df = sales_df.groupby(group_by_column)['Sales'].sum().reset_index()

    # Create donut chart
    fig = px.pie(
        state_group_df,
        names=group_by_column,
        values='Sales',
        title=f"📍Sales (💲) by {group_by_column}",
        hole=0.4,  # makes it a donut
        color_discrete_sequence=px.colors.sequential.Viridis
    )

    # Add $M labels inside
    fig.update_traces(
        text=state_group_df['Sales'].apply(lambda x: f"${x/1e6:.2f} M"),
        textposition='auto',
        textfont=dict(size=10, color='white')
    )

    # Clean layout
    fig.update_layout(
        template="plotly_dark",
        width = 500,   # pixels
        height = 500,   # pixels
        legend=dict(
            orientation="h",
            y=-0.2,
            x=0.5,
            xanchor="center"
        )
    )

    fig.show()

#### 📊 State-wise sales analysis for different demographic groups (kids, women, men, and seniors). 

In [31]:
def generate_group_sales_matrix(sales_df):
    """
    Generates a sales matrix for each group and visualizes it as a stacked bar chart."""

    # Prepare data (same logic as yours)
    state_group_df = sales_df.groupby(['State', 'Group'])['Sales'].sum().reset_index()

    # Create stacked bar chart
    fig = px.bar(
        state_group_df,
        x='State',
        y='Sales',
        color='Group',
        title='💵Sales (💲) by State and Group',
        text=state_group_df['Sales'].apply(lambda x: f"${x/1e6:.2f} M"),
        color_discrete_sequence=px.colors.sequential.Aggrnyl
    )

    # Format Y-axis in $ Millions
    fig.update_layout(
        yaxis_title="Sales ($ Millions)",
        xaxis_title="State",
        yaxis=dict(tickformat=".2s")  # auto scales (K, M, B)
    )

    # Put labels inside bars
    fig.update_traces(textposition='auto')

    # Clean layout
    fig.update_layout(
        template="plotly_dark",
        legend_title="Group",
        width=900,
        height=1000
    )

    fig.show()

#### 📊 Time series analysis

In [32]:
def generate_time_sales_matrix(sales_df, time_freq='DAY'):
    """
    Generates a sales matrix over time and visualizes it as a line chart.
    """
    if time_freq == 'DAY':
        sales_data_df = sales_df.groupby('Date')['Sales'].sum().reset_index()
        mode = 'lines'
        title = '📈 Daily Sales(💲) Trend Over Time'
    elif time_freq == 'WEEK':
        sales_data_df = sales_df.resample('W')['Sales'].sum().reset_index()
        mode = 'lines+markers'
        title = '📈 Weekly Sales(💲) Trend Over Time'
    else:
        raise ValueError("Invalid time frequency. Use 'DAY', 'WEEK', 'MONTH', or 'QUARTER'.")

    # Create line chart
    fig = px.line(
        sales_data_df,
        x='Date',
        y='Sales',
        title=title
    )

    # Format Y-axis in $ Millions
    fig.update_yaxes(
        tickprefix="$",
        tickformat=".2s"   # auto K, M, B
    )

    # Add data labels (optional)
    fig.update_traces(
        text=sales_data_df['Sales'].apply(lambda x: f"${x/1e6:.2f} M"),
        textposition="top center",
        mode=mode, 
        line=dict(color='cyan', width=2),           # line color
        marker=dict(size=10, color='cyan', symbol='circle'),  # same as line
    )

    # Clean layout
    fig.update_layout(
        template="plotly_dark",
        width=900,
        height=500
    )

    fig.show()

In [33]:
def generate_monthly_sales_matrix(sales_df):
    """
    Generates a monthly sales matrix and visualizes it as a bar chart.
    """

    monthly_sales = sales_df.resample('ME')['Sales'].sum().to_frame(name='Sales').reset_index()

    # Create bar chart
    fig = px.bar(
        monthly_sales,
        x='Date',
        y='Sales',
        title='📈 Monthly Sales(💲) Trend Over Time',
        text=monthly_sales['Sales'].apply(lambda x: f"${x/1e6:.2f} M"),  # add labels
        color_discrete_sequence=px.colors.sequential.Viridis
    )

    # Show labels on top of bars
    fig.update_traces(textposition='outside')

    # Format X-axis to Mon-YY
    fig.update_xaxes(
        tickformat="%b-%y",  
        dtick="M1"            # tick every 1 month
    )

    # Format Y-axis in $ Millions
    fig.update_yaxes(
        tickprefix="$",
        tickformat=".2s"  
    )

    # Layout
    fig.update_layout(
        template='plotly_dark',
        width=800,
        height=500,
        showlegend=False  # hide legend if color matches x-axis
    )

    fig.show()

In [34]:
def generate_quarterly_sales_matrix(sales_df):
    """
    Generates a quarterly sales matrix and visualizes it as a bar chart.
    """
    quarterly_sales = sales_df.resample('QE')['Sales'].sum().to_frame(name='Sales').reset_index()

    # Create bar chart
    fig = px.bar(
        quarterly_sales,
        x='Date',
        y='Sales',
        title='📈 Quarterly Sales(💲) Trend Over Time',
        text=quarterly_sales['Sales'].apply(lambda x: f"${x/1e6:.2f} M"),  # add labels
        color_discrete_sequence=px.colors.sequential.Viridis
    )

    # Show labels on top of bars
    fig.update_traces(textposition='outside')

    # Format X-axis to Mon-YY
    fig.update_xaxes(
        tickformat="%b-%y",  
        dtick="M1"           
    )

    # Format Y-axis in $ Millions
    fig.update_yaxes(
        tickprefix="$",
        tickformat=".2s"  
    )

    # Layout
    fig.update_layout(
        template='plotly_dark',
        width=800,
        height=500,
        showlegend=False  # hide legend if color matches x-axis
    )

    fig.show()

In [35]:

def generate_time_sales_by_state_matrix(sales_df, time_freq='DAY'):
    """
    Generates a sales matrix over time with a line chart.
    Supports multiple states as separate lines (legend).
    
    sales_df must have columns: 'Date', 'Sales', 'State'
    """
    
    if time_freq == 'DAY':
        sales_data_df = sales_df.groupby(['Date', 'State'])['Sales'].sum().reset_index()
        mode = 'lines'
        title = '📈 Daily Sales(💲) Trend Over Time by State'
    elif time_freq == 'WEEK':
        # Make sure 'Date' is datetime index for resample
        df_copy = sales_df.copy()
        df_copy['Date'] = pd.to_datetime(df_copy['Date'])
        df_copy.set_index('Date', inplace=True)
        sales_data_df = df_copy.groupby('State').resample('W')['Sales'].sum().reset_index()
        mode = 'lines+markers'
        title = '📈 Weekly Sales(💲) Trend Over Time by State'
    else:
        raise ValueError("Invalid time frequency. Use 'DAY' or 'WEEK'.")
    
    # Create line chart with State as legend
    fig = px.line(
        sales_data_df,
        x='Date',
        y='Sales',
        color='State',          # this adds the legend
        title=title,
        markers=True            # show markers for all points
    )
    
    # Format Y-axis in $ Millions
    fig.update_yaxes(
        tickprefix="$",
        tickformat=".2s"   # auto K, M, B
    )
    
    # Add data labels for each line
    fig.update_traces(
        text=sales_data_df['Sales'].apply(lambda x: f"${x/1e6:.2f} M"),
        textposition="top center",
        line=dict(width=2),
        marker=dict(size=8),
        mode=mode
    )
    
    # Clean layout
    fig.update_layout(
        template="plotly_dark",
        width=900,
        height=500,
        legend_title_text='State'
    )
    
    fig.show()

### 📊 Sales Analysis Dashboard:

In [36]:
# State-wise sales Report
generate_sales_donut_matrix(df, group_by_column='Group')
generate_sales_donut_matrix(df, group_by_column='State')
generate_sales_donut_matrix(df, group_by_column='Time')
generate_group_sales_matrix(df)
generate_time_sales_matrix(df, time_freq='DAY')
generate_time_sales_matrix(df, time_freq='WEEK')
generate_time_sales_by_state_matrix(df, time_freq='DAY')
generate_monthly_sales_matrix(df)
generate_quarterly_sales_matrix(df)


### 📊 Sales Performance Observations
- **Geographic Concentration:** `VIC` is the clear market leader, generating **$105M (31% of total sales)**. `NSW` follows at **$75M (22%)**. Combined, these two states represent over half of all revenue.
- **Underperforming Regions:** `WA`, `NT`, and `TAS` are the lowest contributors, with sales clustering tightly between **$22.15M** and **$22.76M**.
- **Demographic Consistency:** Performance is remarkably uniform across all segments (`Men`, `Women`, `Kids`, and `Seniors`)
- **Temporal Stability:** Sales remain consistent throughout the day, with no significant variance between `Morning`, `Afternoon`, and `Evening` periods.
- **Positive Growth Trend:** There is a strong upward trajectory in total sales, growing from **$114M (Nov-20)** to **$135M (Jan-21)**. This growth is synchronized across all states.

## 💡Recommendations to Increase Sales
### 📈 Stimulate Growth in Low-Volume States
- **Localized Marketing:** `WA`, `NT`, and `TAS` may require region-specific campaigns (e.g., addressing local climate or cultural events) to break the $23M ceiling.
- **Introductory Offers:** Use "first-purchase" discounts specifically targeted at these three states to lower the barrier for new customers.
- **Regional Expansion:** Investigate if there are underserved suburbs within `WA`, `NT`, and `TAS` where new physical outlets or localized digital ads could capture even more of the existing demand.

### 🚀 **Scale Success in High-Performing States**
- **VIC & NSW Loyalty:** Since these states are your "engines," implement loyalty programs or exclusive VIP events here to prevent churn and increase "share of wallet."
- **Regional Expansion:** Investigate if there are underserved suburbs within `VIC` and `NSW` where new physical outlets or localized digital ads could capture even more of the existing demand.
